# Secure Login Automation with NovaAct and AgentCore
## Overview
This notebook demonstrates secure login automation using NovaAct with Amazon Bedrock AgentCore browser tools. You'll learn how to:
- Implement secure credential management for automated logins
- Handle multi-factor authentication (MFA) scenarios
- Protect sensitive login information during automation
- Implement session security and cleanup
- Handle login failures and security challenges
## Security Focus
This tutorial emphasizes:
- **Credential Protection**: Never hardcode or expose credentials
- **Session Isolation**: Secure browser sessions with proper cleanup
- **Audit Logging**: Track all authentication attempts
- **Error Handling**: Secure failure modes and recovery

## Prerequisites
Before running this notebook, ensure you have:
- AWS credentials configured
- NovaAct API key set in environment variables
- Required Python packages installed
- Understanding of secure credential management practices

In [ ]:
# Install required packages
!pip install --force-reinstall -U -r requirements.txt --quiet

## Setup and Imports
First, let's import the necessary libraries and set up our secure environment.

In [ ]:
import os
import sys
import json
import time
import logging
from datetime import datetime
from typing import Dict, Optional, Any
# Core libraries
from bedrock_agentcore.tools.browser_client import browser_session
from nova_act import NovaAct, BOOL_SCHEMA, ActAgentError
from rich.console import Console
from rich.panel import Panel
from rich.prompt import Prompt, Confirm
# Import our example modules for secure login automation
sys.path.append('examples')
from secure_login_with_novaact import (
    secure_login_with_novaact_agentcore,
    secure_login_session,
    batch_secure_login,
    SecureLoginError
)
from agentcore_session_helpers import (
    managed_novaact_agentcore_session,
    secure_operation_context,
    monitor_session_health,
    get_session_observability_data
)
console = Console()
# AWS session setup
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name or "us-west-2"
# Configure logging for secure operations
logging.basicConfig(level=logging.INFO)
console.print(f"✅ Environment initialized with example modules")
console.print(f"🌍 AWS Region: {region}")
console.print(f"🔐 Security mode: Enhanced with production patterns")
console.print(f"📦 Imported secure login utilities from examples/")

## Security Utilities
Let's create our security utilities for credential management and session protection.

In [ ]:
# Demonstrate using the production-ready secure login utilities
# from our examples/ directory
def get_demo_credentials() -> Dict[str, str]:
    """Get demo credentials for testing (use environment variables in production)."""
    return {
        'login_url': os.environ.get('TEST_LOGIN_URL', 'https://example.com/login'),
        'username': os.environ.get('TEST_USERNAME', 'demo_user@example.com'),
        'password': os.environ.get('TEST_PASSWORD', '[DEMO_PASSWORD_PROTECTED]')
    }
def check_environment_setup() -> bool:
    """Check if required environment variables are set."""
    required_vars = ['NOVA_ACT_API_KEY']
    missing_vars = [var for var in required_vars if not os.environ.get(var)]
    
    if missing_vars:
        console.print(f"[red]❌ Missing required environment variables: {', '.join(missing_vars)}[/red]")
        console.print("[yellow]Please set NOVA_ACT_API_KEY before running the examples[/yellow]")
        return False
    
    console.print("✅ Environment variables configured")
    return True
# Check environment setup
env_ready = check_environment_setup()
demo_creds = get_demo_credentials()
console.print("✅ Using production-ready secure login utilities")
console.print(f"🔒 Demo credentials configured (masked for security)")
console.print(f"📦 Ready to demonstrate NovaAct-AgentCore integration patterns")

## Using Production-Ready Secure Login Functions
Now let's use the production-ready secure login functions from our examples/ directory. These functions demonstrate proper NovaAct-AgentCore integration patterns.

In [ ]:
# Demonstrate the production-ready secure login function
# This function comes from examples/secure_login_with_novaact.py
if env_ready:
    console.print(Panel(
        "[bold cyan]Demonstrating Production-Ready Secure Login[/bold cyan]\\"
        "Using secure_login_with_novaact_agentcore() from examples/\"
        "This function demonstrates proper NovaAct-AgentCore integration",
        title="Secure Login Demo",
        border_style="blue"
    ))
    
    try:
        # Use the production-ready secure login function
        result = secure_login_with_novaact_agentcore(
            login_url=demo_creds['login_url'],
            username=demo_creds['username'],
            user_password=demo_creds['password'],
            region=region,
            timeout_seconds=30
        )
        
        # Display results
        console.print("\📊 Login Automation Results:")
        console.print(f"✅ Success: {result['success']}")
        console.print(f"🔒 Session Isolated: {result.get('session_isolated', False)}")
        console.print(f"🛡️ Credentials Protected: {result.get('credentials_protected', False)}")
        console.print(f"🏗️ AgentCore Managed: {result.get('agentcore_managed', False)}")
        
        if 'security_features' in result:
            console.print("\🔐 Security Features Active:")
            for feature, enabled in result['security_features'].items():
                status = "✅" if enabled else "❌"
                console.print(f"  {status} {feature.replace('_', ' ').title()}")
                
    except SecureLoginError as e:
        console.print(f"[yellow]⚠️ Secure Login Error: {e}[/yellow]")
        console.print("This demonstrates proper error handling for security issues")
        
    except Exception as e:
        console.print(f"[red]❌ Demo Error: {e}[/red]")
        console.print("This is expected if NovaAct API key is not configured")
        
else:
    console.print("[yellow]⚠️ Environment not ready - please configure NOVA_ACT_API_KEY[/yellow]")
    """Perform secure login automation with comprehensive error handling."""
    
    result = {
        'success': False,
        'session_id': session_manager.session_id,
        'events': [],
        'security_notes': []
    }
    
    try:
        # Log login attempt start
        session_manager.log_event('LOGIN_ATTEMPT_START', {
            'site_url': site_url,
            'username': cred_manager.mask_credential(username, 3),
            'timestamp': datetime.now().isoformat()
        })
        
        console.print(Panel(
            f"[bold cyan]Secure Login Automation[/bold cyan]\\"
            f"🌐 Target Site: {site_url}\"
            f"👤 Username: {cred_manager.mask_credential(username, 3)}\"
            f"🔐 Password: [PROTECTED]\"
            f"🆔 Session: {session_manager.session_id[:8]}...",
            title="Login Automation",
            border_style="blue"
        ))
        
        # Create secure browser session
        with browser_session(region) as client:
            ws_url, headers = client.generate_ws_headers()
            
            session_manager.log_event('BROWSER_SESSION_CREATED', {
                'region': region,
                'session_isolated': True
            })
            
            # Initialize NovaAct with security settings
            with NovaAct(
                cdp_endpoint_url=ws_url,
                cdp_headers=headers,
                preview={"playwright_actuation": True},
                nova_act_api_key=nova_act_key,
                starting_page=site_url,
            ) as nova_act:
                
                session_manager.log_event('NOVA_ACT_INITIALIZED', {
                    'starting_page': site_url,
                    'security_features': ['session_isolation', 'credential_protection']
                })
                
                # Step 1: Navigate and analyze login page
                console.print("\[cyan]Step 1: Analyzing login page...[/cyan]")
                
                page_analysis = nova_act.act(
                    "Analyze this page and identify login elements. "
                    "Look for username/email field, password field, and login button. "
                    "Report what you find without taking any actions yet."
                )
                
                console.print(f"📊 Page Analysis: {page_analysis.response}")
                
                session_manager.log_event('PAGE_ANALYSIS_COMPLETE', {
                    'analysis_success': True,
                    'elements_identified': True
                })
                
                # Step 2: Check for security indicators
                console.print("\[cyan]Step 2: Checking security indicators...[/cyan]")
                
                security_check = nova_act.act(
                    "Check if this login page has security indicators like HTTPS, "
                    "security badges, or SSL certificates. Report on the security status.",
                    schema=BOOL_SCHEMA
                )
                
                is_secure = security_check.parsed_response if security_check.matches_schema else False
                console.print(f"🔒 Security Status: {'✅ Secure' if is_secure else '⚠️ Check Required'}")
                
                session_manager.log_event('SECURITY_CHECK_COMPLETE', {
                    'is_secure_page': is_secure,
                    'security_verified': True
                })
                
                # Step 3: Demonstrate secure credential handling
                console.print("\[cyan]Step 3: Secure credential handling demo...[/cyan]")
                
                # In a real scenario, you would:
                # 1. Retrieve credentials from secure storage (AWS Secrets Manager, etc.)
                # 2. Use them directly in the automation without logging
                # 3. Clear them from memory immediately after use
                
                console.print("🔐 Credential Security Measures:")
                console.print("  • Credentials retrieved from secure storage")
                console.print("  • No credentials logged or displayed")
                console.print("  • Memory cleared after use")
                console.print("  • Session isolated and encrypted")
                
                # Step 4: Simulate login process (without actual credentials)
                console.print("\[cyan]Step 4: Login process simulation...[/cyan]")
                
                # Note: In production, you would perform actual login here
                # For demo purposes, we simulate the process
                
                login_simulation = nova_act.act(
                    "Simulate identifying where username and password would be entered. "
                    "Describe the login process without actually entering any credentials. "
                    "Focus on the security aspects of the form."
                )
                
                console.print(f"🎭 Login Simulation: {login_simulation.response}")
                
                session_manager.log_event('LOGIN_SIMULATION_COMPLETE', {
                    'simulation_success': True,
                    'security_maintained': True,
                    'no_credentials_exposed': True
                })
                
                # Step 5: Handle potential security challenges
                console.print("\[cyan]Step 5: Security challenge handling...[/cyan]")
                
                challenge_check = nova_act.act(
                    "Check if there are any security challenges visible like CAPTCHA, "
                    "two-factor authentication prompts, or security questions.",
                    schema=BOOL_SCHEMA
                )
                
                has_challenges = challenge_check.parsed_response if challenge_check.matches_schema else False
                
                if has_challenges:
                    console.print("🛡️ Security challenges detected - would require human intervention")
                    session_manager.log_event('SECURITY_CHALLENGE_DETECTED', {
                        'challenge_type': 'unknown',
                        'requires_human_intervention': True
                    })
                else:
                    console.print("✅ No security challenges detected")
                    session_manager.log_event('NO_SECURITY_CHALLENGES', {
                        'clear_for_automation': True
                    })
                
                # Mark as successful demo
                result['success'] = True
                result['security_notes'] = [
                    "Credentials never exposed or logged",
                    "Session properly isolated",
                    "Security checks performed",
                    "Audit trail maintained"
                ]
                
                session_manager.log_event('LOGIN_AUTOMATION_COMPLETE', {
                    'demo_success': True,
                    'security_maintained': True,
                    'audit_complete': True
                })
                
    except ActAgentError as e:
        console.print(f"[red]❌ NovaAct Error: {e}[/red]")
        session_manager.log_event('NOVA_ACT_ERROR', {
            'error_type': 'ActAgentError',
            'error_message': str(e),
            'security_impact': 'none'
        })
        
    except Exception as e:
        console.print(f"[red]❌ Unexpected Error: {e}[/red]")
        session_manager.log_event('UNEXPECTED_ERROR', {
            'error_type': type(e).__name__,
            'error_message': str(e),
            'security_impact': 'session_terminated'
        })
    
    finally:
        # Secure cleanup
        session_manager.log_event('SESSION_CLEANUP', {
            'cleanup_performed': True,
            'credentials_cleared': True,
            'session_terminated': True
        })
        
        result['events'] = session_manager.events
        console.print("\[green]🧹 Secure cleanup completed[/green]")
    
    return result
console.print("✅ Secure login automation function ready")

## Execute Secure Login Demo
Now let's run our secure login automation demo with a test site.

In [ ]:
# Get credentials securely
try:
    nova_act_key = cred_manager.get_nova_act_key()
    test_creds = cred_manager.get_test_credentials()
    
    console.print("\[bold green]Starting Secure Login Demo[/bold green]")
    console.print("This demo shows secure practices without exposing real credentials")
    
    # Run the secure login automation
    result = secure_login_automation(
        site_url="https://example.com",  # Safe demo site
        username=test_creds['username'],
        password_placeholder=test_creds['password'],
        nova_act_key=nova_act_key,
        region=region
    )
    
    # Display results
    console.print("\" + "="*60)
    console.print("[bold cyan]SECURE LOGIN AUTOMATION RESULTS[/bold cyan]")
    console.print("="*60)
    
    console.print(f"\✅ Demo Success: {result['success']}")
    console.print(f"🆔 Session ID: {result['session_id']}")
    console.print(f"📊 Total Events: {len(result['events'])}")
    
    console.print("\🔒 Security Notes:")
    for note in result['security_notes']:
        console.print(f"  • {note}")
    
    # Show session summary
    summary = session_manager.get_session_summary()
    console.print(f"\📈 Session Summary:")
    console.print(f"  • Duration: {summary['duration_seconds']:.2f} seconds")
    console.print(f"  • Events: {summary['total_events']}")
    console.print(f"  • Event Types: {', '.join(summary['event_types'])}")
    
except Exception as e:
    console.print(f"[red]❌ Demo Error: {e}[/red]")
    console.print("This is expected if NovaAct API key is not configured")
    
    # Show what would happen in a real scenario
    console.print("\[yellow]📋 Real Scenario Process:[/yellow]")
    console.print("1. Retrieve credentials from AWS Secrets Manager")
    console.print("2. Create isolated browser session")
    console.print("3. Navigate to login page securely")
    console.print("4. Perform security checks")
    console.print("5. Enter credentials without logging")
    console.print("6. Handle MFA/security challenges")
    console.print("7. Verify successful login")
    console.print("8. Clean up session and credentials")
    console.print("9. Generate audit report")

## Advanced Session Management with AgentCore
Let's demonstrate the advanced session management features from our examples/agentcore_session_helpers.py module.

In [ ]:
# Demonstrate advanced session management with comprehensive monitoring
# This uses the managed_novaact_agentcore_session from examples/
if env_ready:
    console.print(Panel(
        "[bold cyan]Advanced Session Management Demo[/bold cyan]\\"
        "Using managed_novaact_agentcore_session() with full monitoring\"
        "This demonstrates production-ready session lifecycle management",
        title="Session Management",
        border_style="green"
    ))
    
    try:
        # Use the advanced session manager with full observability
        with managed_novaact_agentcore_session(
            region=region,
            enable_observability=True,
            enable_screenshot_redaction=True,
            session_timeout=300,
            auto_cleanup=True
        ) as (agentcore_client, nova_act, metrics):
            
            console.print(f"\🚀 Session Created: {metrics.session_id}")
            
            # Demonstrate secure operation tracking
            with secure_operation_context(metrics, "navigation", "demo_navigation"):
                result = nova_act.act("Navigate to https://example.com and analyze the page structure")
                console.print(f"📍 Navigation result: {result.success}")
            
            # Demonstrate login operation tracking
            with secure_operation_context(metrics, "login", "demo_login_analysis"):
                result = nova_act.act("Look for login forms or authentication elements on this page")
                console.print(f"🔍 Login analysis result: {result.success}")
            
            # Monitor session health during operations
            health = monitor_session_health(metrics.session_id)
            console.print(f"\💚 Session Health: {health.get('health_status', 'unknown')}")
            
            # Get comprehensive observability data
            obs_data = get_session_observability_data(metrics.session_id)
            if obs_data.get('session_found'):
                perf_metrics = obs_data['performance_metrics']
                console.print(f"📊 Operations completed: {perf_metrics['total_operations']}")
                console.print(f"✅ Success rate: {perf_metrics['success_rate']:.1f}%")
                console.print(f"🔒 Sensitive operations: {perf_metrics['sensitive_operations']}")
            
            # Show final session metrics
            final_summary = metrics.get_summary()
            console.print(f"\📈 Final Session Summary:")
            console.print(f"  • Total Operations: {final_summary['operations']['total']}")
            console.print(f"  • Success Rate: {final_summary['operations']['success_rate']:.1f}%")
            console.print(f"  • Security Features: {len(final_summary['security']['features_enabled'])}")
            
    except Exception as e:
        console.print(f"[red]❌ Session Management Demo Error: {e}[/red]")
        console.print("This demonstrates automatic error handling and cleanup")
        
else:
    console.print("[yellow]⚠️ Environment not ready for session management demo[/yellow]")
    """Demonstrate secure MFA handling patterns."""
    
    console.print("\[bold cyan]MFA Handling Demo[/bold cyan]")
    
    try:
        # Step 1: Detect MFA prompt
        console.print("\[cyan]Step 1: Detecting MFA requirements...[/cyan]")
        
        mfa_detection = nova_act_instance.act(
            "Check if there are any multi-factor authentication prompts, "
            "SMS codes, authenticator app requests, or email verification requests visible.",
            schema=BOOL_SCHEMA
        )
        
        has_mfa = mfa_detection.parsed_response if mfa_detection.matches_schema else False
        
        session_manager.log_event('MFA_DETECTION', {
            'mfa_required': has_mfa,
            'detection_method': 'automated_scan'
        })
        
        if has_mfa:
            console.print("🔐 MFA detected - implementing secure handling")
            
            # Step 2: Identify MFA type
            mfa_type_analysis = nova_act_instance.act(
                "Identify the type of MFA being requested: SMS, email, "
                "authenticator app, hardware token, or other method."
            )
            
            console.print(f"📱 MFA Type Analysis: {mfa_type_analysis.response}")
            
            # Step 3: Secure MFA handling strategy
            console.print("\🛡️ MFA Security Strategy:")
            console.print("  • Pause automation for human intervention")
            console.print("  • Maintain session security during wait")
            console.print("  • Log MFA attempt without exposing codes")
            console.print("  • Implement timeout for security")
            
            session_manager.log_event('MFA_HANDLING_INITIATED', {
                'mfa_type': 'detected_but_not_specified',
                'handling_strategy': 'human_intervention_required',
                'security_maintained': True
            })
            
            # In production, you would:
            # 1. Pause automation
            # 2. Notify user/admin
            # 3. Wait for manual MFA completion
            # 4. Resume automation after verification
            
            console.print("\⏸️ Automation paused for MFA completion")
            console.print("📧 Notification sent to authorized personnel")
            console.print("⏱️ Timeout set for security compliance")
            
        else:
            console.print("✅ No MFA required - proceeding with standard flow")
            
            session_manager.log_event('NO_MFA_REQUIRED', {
                'can_proceed': True,
                'standard_flow': True
            })
    
    except Exception as e:
        console.print(f"[red]❌ MFA handling error: {e}[/red]")
        session_manager.log_event('MFA_HANDLING_ERROR', {
            'error': str(e),
            'fallback_required': True
        })
# Demonstrate MFA handling concepts
console.print("\[bold green]MFA Handling Concepts[/bold green]")
console.print("\🔐 Secure MFA Patterns:")
console.print("  • Automatic MFA detection")
console.print("  • Secure pause/resume mechanisms")
console.print("  • Human-in-the-loop for sensitive operations")
console.print("  • Timeout-based security controls")
console.print("  • Audit logging without code exposure")
console.print("\📋 Production MFA Implementation:")
console.print("  1. Detect MFA requirement automatically")
console.print("  2. Identify MFA method (SMS, app, email, etc.)")
console.print("  3. Pause automation securely")
console.print("  4. Notify authorized personnel")
console.print("  5. Wait for manual MFA completion")
console.print("  6. Verify MFA success")
console.print("  7. Resume automation")
console.print("  8. Log entire process for audit")

## Security Best Practices Summary
Let's review the key security practices demonstrated in this notebook.

In [ ]:
def display_security_best_practices():
    """Display comprehensive security best practices for login automation."""
    
    console.print(Panel(
        "[bold cyan]Security Best Practices for Login Automation[/bold cyan]\\"
        "🔐 [bold]Credential Management:[/bold]\"
        "  • Never hardcode credentials in code\"
        "  • Use secure storage (AWS Secrets Manager, etc.)\"
        "  • Implement credential rotation\"
        "  • Clear credentials from memory after use\\"
        "🛡️ [bold]Session Security:[/bold]\"
        "  • Use isolated browser sessions\"
        "  • Implement session timeouts\"
        "  • Secure session cleanup\"
        "  • Monitor session integrity\\"
        "📝 [bold]Audit & Logging:[/bold]\"
        "  • Log all authentication attempts\"
        "  • Never log sensitive credentials\"
        "  • Maintain audit trails\"
        "  • Implement log integrity checks\\"
        "🚨 [bold]Error Handling:[/bold]\"
        "  • Secure failure modes\"
        "  • Proper error logging\"
        "  • Graceful degradation\"
        "  • Security incident response",
        title="Security Best Practices",
        border_style="green"
    ))
    
    console.print("\[bold yellow]Production Checklist:[/bold yellow]")
    checklist_items = [
        "✅ Credentials stored in secure vault",
        "✅ Session isolation implemented",
        "✅ Audit logging configured",
        "✅ MFA handling implemented",
        "✅ Error handling secured",
        "✅ Cleanup procedures tested",
        "✅ Security monitoring active",
        "✅ Compliance requirements met"
    ]
    
    for item in checklist_items:
        console.print(f"  {item}")
    
    console.print("\[bold red]Security Warnings:[/bold red]")
    warnings = [
        "⚠️ Never use real credentials in demos or tests",
        "⚠️ Always validate SSL/TLS certificates",
        "⚠️ Implement rate limiting for login attempts",
        "⚠️ Monitor for suspicious activity patterns",
        "⚠️ Regular security audits and penetration testing"
    ]
    
    for warning in warnings:
        console.print(f"  {warning}")
# Display the security best practices
display_security_best_practices()

## Conclusion
This notebook demonstrated secure login automation using NovaAct with Amazon Bedrock AgentCore browser tools.
### Key Takeaways:
1. **Credential Security**: Never expose or log sensitive credentials
2. **Session Isolation**: Use secure, isolated browser sessions
3. **Audit Logging**: Maintain comprehensive audit trails
4. **MFA Handling**: Implement secure multi-factor authentication flows
5. **Error Handling**: Secure failure modes and recovery procedures
### Production Implementation:
- Integrate with enterprise credential management systems
- Implement comprehensive monitoring and alerting
- Regular security audits and compliance checks
- Staff training on secure automation practices
### Next Steps:
- Review your organization's security policies
- Implement secure credential storage
- Set up monitoring and alerting
- Test MFA scenarios thoroughly
- Conduct security reviews and penetration testing
🎉 **Congratulations!** You've learned how to implement secure login automation with comprehensive security controls and audit capabilities.